In [2]:
import pandas as pd
import zipfile
import os

with zipfile.ZipFile('../data/raw/census/tx2020.pl.zip', 'r') as z:
    z.extractall('../data/raw/census/')

print("Files extracted successfully")
print(os.listdir('../data/raw/census/'))

Files extracted successfully
['tx2020.pl.zip', 'tx000022020.pl', '.gitkeep', 'tx000032020.pl', 'tx000012020.pl', 'txgeo2020.pl']


In [4]:
# Read the geographic header file
# This file tells us the geography of each row (state, county, tract, block)
geo_headers = [
    'FILEID', 'STUSAB', 'SUMLEV', 'GEOVAR', 'GEOCOMP', 'CHARITER', 'CIFSN', 'LOGRECNO',
    'GEOID', 'GEOCODE', 'REGION', 'DIVISION', 'STATE', 'STATENS', 'COUNTY', 'COUNTYCC',
    'COUNTYNS', 'COUSUB', 'COUSUBCC', 'COUSUBNS', 'SUBMCD', 'SUBMCDCC', 'SUBMCDNS',
    'ESTATE', 'ESTATECC', 'ESTATENS', 'CONCIT', 'CONCITCC', 'CONCITNS', 'PLACE',
    'PLACECC', 'PLACENS', 'TRACT', 'BLKGRP', 'BLOCK', 'AIANHH', 'AIHHTLI', 'AIANHHFP',
    'AIANHHCC', 'AIANHHNS', 'AITS', 'AITSCC', 'AITSFP', 'AITSNS', 'TTRACT', 'TBLKGRP',
    'ANRC', 'ANRCCC', 'ANRCNS', 'CBSA', 'MEMI', 'CSA', 'METDIV', 'NECTA', 'NMEMI',
    'NECTADIV', 'CNECTA', 'CBSAPCI', 'NECTAPCI', 'UA', 'UATYPE', 'UR', 'CD116',
    'CD118', 'CD119', 'CD120', 'CD121', 'SLDU18', 'SLDU22', 'SLDU24', 'SLDL18',
    'SLDL22', 'SLDL24', 'VTD', 'VTDI', 'ZCTA', 'SDELM', 'SDSEC', 'SDUNI', 'PUMA',
    'AREALAND', 'AREAWATR', 'BASENAME', 'NAME', 'FUNCSTAT', 'GCUNI', 'POP100',
    'HU100', 'INTPTLAT', 'INTPTLON', 'LSADC', 'PARTFLAG', 'RESERVE2', 'RESERVE3',
    'RESERVE4', 'MSACSA', 'BEALE', 'NMEMI2', 'UACE', 'CBSATYPE'
]

geo = pd.read_csv('../data/raw/census/txgeo2020.pl', 
                  sep='|', 
                  header=None, 
                  names=geo_headers,
                  dtype=str,
                  low_memory=False)

print(f"Total rows: {len(geo)}")

# Filter to Travis County (FIPS 453) and block level (SUMLEV 750)
travis_blocks = geo[(geo['COUNTY'] == '453') & (geo['SUMLEV'] == '750')]
print(f"Travis County blocks: {len(travis_blocks)}")
travis_blocks[['LOGRECNO', 'COUNTY', 'TRACT', 'BLOCK', 'POP100']].head(10)

Total rows: 943920
Travis County blocks: 16906


,LOGRECNO,COUNTY,TRACT,BLOCK,POP100
869787,0869788,453,000101,1000,1000
869788,0869789,453,000101,1001,1001
869789,0869790,453,000101,1002,1002
869790,0869791,453,000101,1003,1003
869791,0869792,453,000101,1004,1004
869792,0869793,453,000101,1005,1005
869793,0869794,453,000101,1006,1006
869794,0869795,453,000101,1007,1007
869795,0869796,453,000101,1008,1008
869796,0869797,453,000101,1009,1009


In [5]:
# Read the population table (table 1)
# This contains the actual population counts
pop = pd.read_csv('../data/raw/census/tx000012020.pl',
                  sep='|',
                  header=None,
                  dtype=str,
                  low_memory=False)

print(f"Shape: {pop.shape}")
print(pop.head(3))

Shape: (943920, 149)
    0   1    2   3        4         5         6         7        8       9    \
0  PLST  TX  000  01  0000001  29145505  24011767  14609365  3552997  278948   
1  PLST  TX  000  01  0000002     57922     54327     34922    11577     357   
2  PLST  TX  000  01  0000003     18610     14756     10909      237     220   

   ... 139  140  141  142 143 144 145 146 147 148  
0  ...   0  581  305  203  30  21  16   6  58  58  
1  ...   0    2    0    2   0   0   0   0   3   3  
2  ...   0    0    0    0   0   0   0   0   0   0  

[3 rows x 149 columns]


In [6]:
# The key columns we need are:
# Column 4: LOGRECNO (links to geo file)
# Column 5: P0010001 (total population)

pop_clean = pop[[4, 5]].copy()
pop_clean.columns = ['LOGRECNO', 'total_pop']
pop_clean['total_pop'] = pd.to_numeric(pop_clean['total_pop'], errors='coerce')

# Now join to the Travis County blocks from the geo file
travis_blocks_reset = travis_blocks.reset_index()
travis_pop = travis_blocks_reset.merge(pop_clean, on='LOGRECNO', how='left')

print(f"Travis County blocks with population: {len(travis_pop)}")
print(f"Total Travis County population: {travis_pop['total_pop'].sum():,.0f}")
travis_pop[['COUNTY', 'TRACT', 'BLOCK', 'total_pop']].head(10)

Travis County blocks with population: 16906
Total Travis County population: 1,290,188


,COUNTY,TRACT,BLOCK,total_pop
0,453,000101,1000,50
1,453,000101,1001,0
2,453,000101,1002,54
3,453,000101,1003,61
4,453,000101,1004,205
5,453,000101,1005,96
6,453,000101,1006,58
7,453,000101,1007,76
8,453,000101,1008,124
9,453,000101,1009,40


In [7]:
# Save Travis County block population data
travis_pop[['COUNTY', 'TRACT', 'BLOCK', 'LOGRECNO', 'total_pop']].to_csv(
    '../data/processed/travis_census_blocks_2020.csv', 
    index=False
)

print("Saved to /data/processed/travis_census_blocks_2020.csv")

Saved to /data/processed/travis_census_blocks_2020.csv


In [8]:
notes = """# Census Data Notes

## 2020 Census P.L. 94-171 Redistricting File — Texas
- **Source:** U.S. Census Bureau
- **URL:** https://www2.census.gov/programs-surveys/decennial/2020/data/01-Redistricting_File--PL_94-171/Texas/
- **File:** tx2020.pl.zip (95MB)
- **Download date:** May 2026
- **Key file:** tx000012020.pl (population table), txgeo2020.pl (geography)
- **Population field:** Column 5 (P0010001) = total population
- **Geography field:** LOGRECNO links population table to geo file

## Travis County Summary
- **FIPS code:** 453
- **Summary level for blocks:** SUMLEV = 750
- **Total blocks:** 16,906
- **Total population:** 1,290,188
- **Processed file:** /data/processed/travis_census_blocks_2020.csv

## Notes
- Population counts are exact 2020 Census counts (not estimates)
- Some blocks have zero population (commercial areas, parks, etc.)
- This data was designed specifically for redistricting work
"""

with open('../data/raw/census/census_notes.md', 'w') as f:
    f.write(notes)

print("census_notes.md saved")

census_notes.md saved


In [9]:
import zipfile
with zipfile.ZipFile('../data/raw/census/Blocks_Pop.zip', 'r') as z:
    print(z.namelist())

['Blocks_Pop.txt']


In [10]:
with zipfile.ZipFile('../data/raw/census/Blocks.zip', 'r') as z:
    print(z.namelist()[:10])

['Blocks.cpg', 'Blocks.dbf', 'Blocks.prj', 'Blocks.sbn', 'Blocks.sbx', 'Blocks.shp', 'Blocks.shp.xml', 'Blocks.shx']


In [11]:
import geopandas as gpd

# Load the blocks shapefile
blocks = gpd.read_file('zip://../data/raw/census/Blocks.zip!Blocks.shp')

print(f"Total blocks statewide: {len(blocks)}")
print(f"Columns: {list(blocks.columns)}")
print(f"CRS: {blocks.crs}")

Total blocks statewide: 668757
Columns: ['BG', 'State', 'CNTY', 'TRT', 'BLK', 'SCTBKEY', 'vtd', 'blkkey', 'CTBKEY', 'CNTYKEY', 'Shape_area', 'Shape_len', 'geometry']
CRS: PROJCS["NAD_1983_Lambert_Conformal_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4269"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",31.1666666666667],PARAMETER["central_meridian",-100],PARAMETER["standard_parallel_1",27.4166666666667],PARAMETER["standard_parallel_2",34.9166666666667],PARAMETER["false_easting",1000000],PARAMETER["false_northing",1000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [12]:
# Filter to Travis County
travis_blocks_geo = blocks[blocks['CNTY'] == 453].copy()
print(f"Travis County blocks: {len(travis_blocks_geo)}")

# Load the population text file
pop_txt = pd.read_csv('../data/raw/census/Blocks_Pop.txt', 
                      sep='|',
                      dtype=str)
print(f"\nPopulation file columns: {list(pop_txt.columns)}")
print(pop_txt.head(3))

Travis County blocks: 0


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/census/Blocks_Pop.txt'